In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score,mean_absolute_error, mean_squared_error
from transformers import RobertaTokenizer, RobertaModel
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from torch.utils.data import DataLoader, Subset
from scipy.stats import pearsonr
from tqdm import tqdm
from sklearn.exceptions import FitFailedWarning
import warnings
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
import matplotlib.pyplot as plt
from scipy.stats import norm
import matplotlib
from sklearn.base import clone


In [2]:
data=pd.read_excel("./algae_unique.xlsx")
data=data.dropna()
smiles_data = data['SMILES_Canonical_RDKit'].tolist()
mgperL = data['mgperL'].values
Duration_Value= data['Duration_Value'].values

In [3]:
mgperL=np.log1p(mgperL)

In [4]:
# 检查需要One-Hot编码的列，并进行编码（如果类别超过一种）
def encode_column(data, column_name):
    unique_values = data[column_name].unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(data[[column_name]])
    else:
        return None  # 只有一种类别时忽略

# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(data, 'effect')
endpoint_encoded = encode_column(data, 'endpoint')
species_encoded = encode_column(data, 'species_group')

# 将需要的列拼接成输入 X
extra_features = data['Duration_Value'].values.reshape(-1, 1)

# 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded, species_encoded]:
    if encoded_feature is not None:
        extra_features = np.hstack((extra_features, encoded_feature))

In [5]:
extra_features.shape

(9045, 3)

In [10]:
extra_dim = extra_features.shape[1]

In [11]:
extra_dim = extra_features.shape[1]
data_extra_features = extra_features

In [12]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import BertTokenizerFast, BertModel
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import optuna

# -------------------------------
# 1. 数据增强与 Dataset 修改：支持额外特征 extra_features
# -------------------------------
def augment_smiles(smiles):
    """简单的数据增强方法：50% 的概率翻转 SMILES 字符串"""
    if random.random() > 0.5:
        return smiles[::-1]
    return smiles

class SMILES_Dataset(Dataset):
    def __init__(self, smiles, reg_labels, extra_features=None, use_augmentation=False):
        self.smiles = smiles
        self.reg_labels = reg_labels
        self.extra_features = extra_features  # 额外特征，要求为数组或列表（每个元素是一个数值向量）
        self.use_augmentation = use_augmentation

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        s = self.smiles[idx]
        if self.use_augmentation:
            s = augment_smiles(s)
        reg_label = self.reg_labels[idx]
        tokens = tokenizer(s, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        # squeeze 扁平化 batch 维度（例如变为 [seq_len]）
        tokens = {key: val.squeeze(0) for key, val in tokens.items()}
        if self.extra_features is not None:
            extra_feat = self.extra_features[idx]
            extra_feat = torch.tensor(extra_feat, dtype=torch.float32)
            return tokens, torch.tensor(reg_label, dtype=torch.float32), extra_feat
        else:
            return tokens, torch.tensor(reg_label, dtype=torch.float32)


In [13]:

# -------------------------------
# 2. 加载本地 BERT 模型（BERT for SMILES）
# -------------------------------
# 请将 checkpoint 指向你的本地模型目录（此处示例使用 'unikei/bert-base-smiles'）
checkpoint = '../models/base_bert'
tokenizer = BertTokenizerFast.from_pretrained(checkpoint)
bert_model = BertModel.from_pretrained(checkpoint)

# -------------------------------
# 3. 模型定义：BERT_Regression
# -------------------------------
class Bert_Regression(nn.Module):
    def __init__(self, dropout_rate, fc1_size, fc2_size, fc3_size, extra_dim=0):
        """
        :param extra_dim: 额外特征维度，如果为 0 则不拼接额外特征
        """
        super(Bert_Regression, self).__init__()
        self.bert = bert_model  # 使用加载好的 BERT 模型
        hidden_size = self.bert.config.hidden_size  # 通常为768
        self.extra_dim = extra_dim
        # 拼接后的维度
        input_dim = hidden_size + extra_dim
        self.regressor = nn.Sequential(
            nn.Linear(input_dim, fc1_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc1_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc1_size, fc2_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc2_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc2_size, fc3_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc3_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc3_size, 1),
            nn.Softplus()
        )
    
    def forward(self, tokens, extra_features=None):
        outputs = self.bert(**tokens)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token 嵌入，形状 [batch_size, 768]
        if self.extra_dim > 0 and extra_features is not None:
            # 拼接额外特征，要求 extra_features 的形状为 [batch_size, extra_dim]
            x = torch.cat([cls_embedding, extra_features], dim=1)
        else:
            x = cls_embedding
        reg_output = self.regressor(x)
        return reg_output

In [14]:

data_smiles = smiles_data
data_labels = mgperL

# 使用 SMILES 作为组依据，确保同一 SMILES 不出现在不同折中
groups = data_smiles

# 如果数据量较大，可根据需要提前封装为 Dataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [15]:
from torch.cuda.amp import autocast, GradScaler


In [17]:
import os
import pickle
import numpy as np
import optuna
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

# 假设以下变量已定义：
# data_smiles, data_labels, data_extra_features, groups, device, extra_dim
# Bert_Regression, SMILES_Dataset

os.makedirs("./optuna_model_set_1", exist_ok=True)
os.makedirs("./optuna_model_set_2", exist_ok=True)

model_sets = ["./optuna_model_set_1", "./optuna_model_set_2"]
set_errors = [float('inf'), float('inf')]
model_save_counter = 0  # 控制保存路径的交替轮换

# =============== 超参数搜索目标函数 ================
def objective(trial):
    global model_save_counter, model_sets, set_errors

    dropout_rate  = trial.suggest_float('dropout_rate', 0.1, 0.5)
    fc1_size      = trial.suggest_int('fc1_size', 256, 1024, step=64)
    fc2_size      = trial.suggest_int('fc2_size', 128, 512, step=32)
    fc3_size      = trial.suggest_int('fc3_size', 64, 256, step=32)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True)
    weight_decay  = trial.suggest_float('weight_decay', 0.0, 0.1)

    gkf = GroupKFold(n_splits=10)
    fold_errors = []
    model_set_dir = model_sets[model_save_counter % 2]

    for fold, (train_idx, val_idx) in enumerate(tqdm(gkf.split(data_smiles, data_labels, groups), total=10, desc=f"Trial {trial.number}")):
        train_smiles = [data_smiles[i] for i in train_idx]
        val_smiles = [data_smiles[i] for i in val_idx]
        train_labels = [data_labels[i] for i in train_idx]
        val_labels = [data_labels[i] for i in val_idx]
        train_extra = data_extra_features[train_idx]
        val_extra = data_extra_features[val_idx]

        train_dataset = SMILES_Dataset(train_smiles, train_labels, extra_features=train_extra)
        val_dataset = SMILES_Dataset(val_smiles, val_labels, extra_features=val_extra)

        train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, drop_last=True, num_workers=8, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=8, pin_memory=True)

        model = Bert_Regression(dropout_rate, fc1_size, fc2_size, fc3_size, extra_dim=extra_dim).to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        criterion = torch.nn.L1Loss()
        scaler = GradScaler()

        for epoch in range(30):
            model.train()
            for tokens, labels_tensor, extra_feats in train_loader:
                tokens = {k: v.to(device) for k, v in tokens.items()}
                extra_feats = extra_feats.to(device)
                labels_tensor = labels_tensor.to(device)

                optimizer.zero_grad()
                with autocast():
                    outputs = model(tokens, extra_feats)
                    loss = criterion(outputs.squeeze(), labels_tensor)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        # 验证并保存模型
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for tokens, labels_tensor, extra_feats in val_loader:
                tokens = {k: v.to(device) for k, v in tokens.items()}
                extra_feats = extra_feats.to(device)
                labels_tensor = labels_tensor.to(device)
                outputs = model(tokens, extra_feats)
                preds.extend(outputs.squeeze().cpu().numpy())
                targets.extend(labels_tensor.cpu().numpy())

        error = np.maximum(targets, preds) / np.minimum(targets, preds)
        median_error = np.median(error)
        fold_errors.append(median_error)

        torch.save(model.state_dict(), f"{model_set_dir}/bert_reg_fold_{fold+1}.pth")

    avg_median_error = np.mean(fold_errors)
    set_errors[model_save_counter % 2] = avg_median_error

    # 判断是否更新替换文件夹指针
    if set_errors[0] > set_errors[1]:
        model_save_counter = 0
    else:
        model_save_counter = 1

    return avg_median_error

# ================ 启动 Optuna 搜索 ===================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

# 输出结果
best_trial = study.best_trial
best_dir = model_sets[0] if set_errors[0] < set_errors[1] else model_sets[1]
print("Best hyperparameters:", best_trial.params)
print("Best model set directory:", best_dir)
print("Best cross-validated median error:", min(set_errors))


[I 2025-04-25 20:29:48,965] A new study created in memory with name: no-name-374242ac-db48-4402-96fd-0ef15e6acb9c
Trial 0: 100%|██████████| 10/10 [41:17<00:00, 247.80s/it]
[I 2025-04-25 21:11:06,953] Trial 0 finished with value: 1.8770825862884521 and parameters: {'dropout_rate': 0.34951242331585797, 'fc1_size': 832, 'fc2_size': 128, 'fc3_size': 192, 'learning_rate': 2.8162953376994645e-05, 'weight_decay': 0.0031782336466101625}. Best is trial 0 with value: 1.8770825862884521.
Trial 1: 100%|██████████| 10/10 [41:14<00:00, 247.47s/it]
[I 2025-04-25 21:52:21,691] Trial 1 finished with value: 2.4570844173431396 and parameters: {'dropout_rate': 0.19428105150501757, 'fc1_size': 768, 'fc2_size': 160, 'fc3_size': 192, 'learning_rate': 1.7066001419799973e-05, 'weight_decay': 0.08090097145384162}. Best is trial 0 with value: 1.8770825862884521.
Trial 2: 100%|██████████| 10/10 [41:17<00:00, 247.73s/it]
[I 2025-04-25 22:33:38,986] Trial 2 finished with value: 2.4654500484466553 and parameters: {'

Best hyperparameters: {'dropout_rate': 0.30292377458448927, 'fc1_size': 1024, 'fc2_size': 224, 'fc3_size': 224, 'learning_rate': 1.9860847200804864e-05, 'weight_decay': 0.00013302077942116408}
Best model set directory: ./optuna_model_set_1
Best cross-validated median error: 1.7338644


In [ ]:
#smiles_embeddings=np.load('./embedding/fish_EC10_smiles_embeddings.npy')

In [ ]:
#val_r2 = r2_score(np.expm1(all_labels), np.expm1(all_preds))  # 还原 log1p 的值